# Notebook 2: KV Cache Quantization
Inspired by KVQuant (Hooper et al., 2024)
Tests FP16, INT8, INT4 quantization across sequence lengths.

In [ ]:
!pip install transformers bitsandbytes accelerate -q

In [ ]:
import torch
import time
import csv
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "gpt2-medium"
os.makedirs("results/tables", exist_ok=True)

seq_lengths = [128, 256, 512]
quant_configs = [None, 8, 4]
quant_labels = ["FP16", "INT8", "INT4"]
rows = []

for bits, label in zip(quant_configs, quant_labels):
    print(f"Loading {label}...")
    if bits == 8:
        config = BitsAndBytesConfig(load_in_8bit=True)
        m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=config, device_map="auto")
    elif bits == 4:
        config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
        m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=config, device_map="auto")
    else:
        m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16, device_map="auto")

    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    tok.pad_token = tok.eos_token

    for seq_len in seq_lengths:
        ids = tok("Hello world", return_tensors="pt")["input_ids"].to("cuda")
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            out = m.generate(input_ids=ids, max_new_tokens=seq_len,
                             do_sample=False, pad_token_id=tok.eos_token_id)
        torch.cuda.synchronize()
        t1 = time.perf_counter()
        peak_mem = round(torch.cuda.max_memory_allocated() / 1e6, 2)
        lat = round((t1 - t0) * 1000, 2)
        toks = out.shape[1] - ids.shape[1]
        tput = round(toks / (t1 - t0), 2)
        row = {"quant": label, "seq_len": seq_len, "peak_mem_mb": peak_mem,
               "latency_ms": lat, "throughput_tok_s": tput}
        rows.append(row)
        print(row)
    del m
    torch.cuda.empty_cache()

with open("results/tables/quant_results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)
print("Saved quant_results.csv")